In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import glob
import os
import cv2
from typing import Optional, Tuple
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
def load_sampled_frames(video_path: str, n_frames: int) -> list[Image.Image]:
    """Load `n_frames` equally spaced as PIL Images."""
    if isinstance(video_path, str) and video_path.endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            cap.release()
            return []

        frame_indices = set(
            np.linspace(0, total_frames - 1, num=n_frames, dtype=int)
        )

        frames = []
        current_idx = 0
        while cap.isOpened() and len(frames) < n_frames:
            ret, frame = cap.read()
            if not ret:
                break
            if current_idx in frame_indices:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # NumPy array to PIL Image
                frames.append(Image.fromarray(rgb_frame))
            current_idx += 1

        cap.release()
        return frames

    image_paths = glob.glob(os.path.join(video_path, "*.jpg"))
    if not image_paths:
        return []

    try:
        image_paths.sort(
            key=lambda p: int(os.path.splitext(os.path.basename(p))[0])
        )
    except ValueError:
        image_paths.sort()

    total_images = len(image_paths)
    sample_indices = np.linspace(
        0, total_images - 1, num=min(n_frames, total_images), dtype=int
    )

    frames = []
    for idx in sample_indices:
        # Load directly as PIL Image
        img = Image.open(image_paths[idx]).convert("RGB")
        frames.append(img)

    return frames

In [ ]:
def remove_letterbox(image: Image.Image, threshold: float = 0.04) -> Image.Image:
    """Detects and crops uniform black letterbox borders from a PIL image."""
    gray = np.array(image.convert("L"), dtype=np.float32)
    if gray.max() > 1.0:
        gray /= 255.0

    non_black = np.where(gray > threshold)
    if non_black[0].size > 0:
        y_min, y_max = int(non_black[0].min()), int(non_black[0].max())
        x_min, x_max = int(non_black[1].min()), int(non_black[1].max())
        return image.crop((x_min, y_min, x_max + 1, y_max + 1))
    return image


def prepare_scaled_tensor(
    image: Image.Image,
    processor: AutoImageProcessor,
    scale: float = 4.0,
    patch_size: int = 16,
    device: str = "cuda",
) -> Tuple[torch.Tensor, Tuple[int, int], Tuple[int, int]]:
    """Rescales an image to a patch-aligned resolution and prepares pixel tensors."""
    orig_w, orig_h = image.size
    scaled_w = int(round(orig_w * scale / patch_size) * patch_size)
    scaled_h = int(round(orig_h * scale / patch_size) * patch_size)

    scaled_img = image.resize((scaled_w, scaled_h), Image.Resampling.BICUBIC)
    inputs = processor(images=scaled_img, do_resize=False, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    h_patches = scaled_h // patch_size
    w_patches = scaled_w // patch_size

    return pixel_values, (h_patches, w_patches), (orig_w, orig_h)


def load_backbone(
    model_id: str, device: str = "cuda"
) -> Tuple[AutoModel, AutoImageProcessor]:
    """Loads and caches the ViT model and corresponding image processor."""
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).eval().to(device)
    return model, processor


def extract_patch_tokens(
    model: AutoModel, pixel_values: torch.Tensor, num_patches: int
) -> torch.Tensor:
    """Runs a forward pass and extracts L2-normalized spatial patch tokens."""
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)

    tokens = outputs.last_hidden_state[0, -num_patches:, :]
    return F.normalize(tokens, p=2, dim=-1)


def compute_anomaly_heatmap(
    tokens: torch.Tensor,
    grid_shape: Tuple[int, int],
    target_shape: Tuple[int, int],
    max_sample_pool: int = 1500,
    border_margin_pct: float = 0.05,
) -> np.ndarray:
    """
    Calculates cosine-distance anomaly scores, suppresses border tokens,
    and bicubic-upsamples back to the target pixel resolution.
    """
    h_patches, w_patches = grid_shape
    num_patches = h_patches * w_patches
    orig_w, orig_h = target_shape

    # Contrast against sampled reference pool to avoid O(N^2) memory bottlenecks
    if num_patches > (max_sample_pool * 2):
        sample_idx = torch.randperm(num_patches)[:max_sample_pool]
        similarity_matrix = torch.matmul(tokens, tokens[sample_idx].T)
    else:
        similarity_matrix = torch.matmul(tokens, tokens.T)

    mean_sim = similarity_matrix.mean(dim=1)
    anomaly_scores = (1.0 - mean_sim).reshape(h_patches, w_patches)

    # ViT border artifact suppression
    border_y = max(1, int(h_patches * border_margin_pct))
    border_x = max(1, int(w_patches * border_margin_pct))
    clean_scores = anomaly_scores.clone()
    fill_val = clean_scores.min()

    clean_scores[:border_y, :] = fill_val
    clean_scores[-border_y:, :] = fill_val
    clean_scores[:, :border_x] = fill_val
    clean_scores[:, -border_x:] = fill_val

    # Upsample to target spatial dimensions
    anomaly_tensor = clean_scores.unsqueeze(0).unsqueeze(0)
    upsampled = (
        F.interpolate(
            anomaly_tensor, size=(orig_h, orig_w), mode="bicubic", align_corners=False
        )
        .squeeze()
        .cpu()
        .numpy()
    )

    # Min-max normalization
    norm_score = (upsampled - upsampled.min()) / (
        upsampled.max() - upsampled.min() + 1e-8
    )
    return norm_score


def segment_anomalies(
    norm_score: np.ndarray,
    percentile_threshold: float = 99.5,
    adaptive_block_size: int = 21,
    morph_kernel_size: int = 3,
) -> np.ndarray:
    """Applies global percentile filtering, local adaptive edge matching, and morphology."""
    
    # Top-tail percentile mask
    threshold_val = np.percentile(norm_score, percentile_threshold)
    binary_mask = (norm_score >= threshold_val).astype(np.uint8) * 255

    # Local adaptive contrast refinement
    roi = (norm_score * 255).astype(np.uint8)
    core_mask = cv2.adaptiveThreshold(
        roi, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, adaptive_block_size, -2
    )
    fine_mask = cv2.bitwise_and(binary_mask, core_mask)

    # Morphological noise removal
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size)
    )
    return cv2.morphologyEx(fine_mask, cv2.MORPH_OPEN, kernel)


def plot_detection_results(
    image: Image.Image,
    norm_score: np.ndarray,
    final_mask: np.ndarray,
    overlay_color: Tuple[int, int, int] = (255, 30, 30),
    alpha: float = 0.5,
) -> None:
    """Renders side-by-side verification subplots."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(image)
    axes[0].set_title("Cropped Input")
    axes[0].axis("off")

    axes[1].imshow(norm_score, cmap="magma")
    axes[1].set_title("DINO Heatmap (Border-Suppressed)")
    axes[1].axis("off")

    img_np = np.array(image).copy()
    overlay = img_np.copy()
    overlay[final_mask == 255] = overlay_color
    blended = cv2.addWeighted(img_np, 1 - alpha, overlay, alpha, 0)

    axes[2].imshow(blended)
    axes[2].set_title("High-Detail Silhouette Mask")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

def run_dino_detector(
    raw_image: Image.Image,
    model: AutoModel,
    processor: AutoImageProcessor,
    resolution_scale: float = 4.0,
    percentile_threshold: float = 99.5,
    visualize: bool = True,
    device: Optional[str] = None,
) -> Tuple[np.ndarray, np.ndarray, Image.Image]:
    """
    End-to-end execution pipeline for ViT-based anomaly segmentation.
    Returns: (final_mask, norm_score, cropped_image)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    cropped_img = remove_letterbox(raw_image)
    pixel_values, (h_patches, w_patches), orig_shape = prepare_scaled_tensor(
        cropped_img, processor, scale=resolution_scale, device=device
    )

    tokens = extract_patch_tokens(
        model, pixel_values, num_patches=h_patches * w_patches
    )
    norm_score = compute_anomaly_heatmap(
        tokens, grid_shape=(h_patches, w_patches), target_shape=orig_shape
    )
    final_mask = segment_anomalies(
        norm_score, percentile_threshold=percentile_threshold
    )

    return final_mask, norm_score, cropped_img

In [ ]:
from typing import Dict, List, Literal, Tuple
import cv2
import numpy as np


def extract_salient_regions(
    binary_mask: np.ndarray,
    method: Literal["otsu", "iqr"] = "otsu",
    iqr_k: float = 1.5,
    min_absolute_pixels: int = 20,
) -> Tuple[np.ndarray, List[Dict]]:
    """
    Filtra componentes pequeños de una máscara binaria y extrae puntos interiores
    garantizados (polos de inaccesibilidad) aptos para geometrías complejas/toros.

    Parámetros:
    -----------
    binary_mask : np.ndarray
        Máscara binaria uint8 (0 y 255), como la retornada por segment_anomalies().
    method : 'otsu' | 'iqr'
        - 'otsu': Aplica Otsu sobre log(1 + área). Ideal cuando coexisten ruido de fondo
                  y regiones de interés evidentes.
        - 'iqr': Regla de outliers Q3 + k*IQR. Ideal si la máscara casi solo contiene ruido
                 y pocas manchas grandes aisladas.
    iqr_k : float
        Multiplicador para IQR (típicamente 1.5 para outliers estándar, 0 para >= Q3).
    min_absolute_pixels : int
        Filtro de corte duro previo para descartar artefactos insignificantes de 1-N píxeles.

    Retorna:
    --------
    filtered_mask : np.ndarray
        Máscara limpia que contiene únicamente las regiones seleccionadas.
    regions_info : List[Dict]
        Lista con metadatos de cada región válida:
        - 'label': ID del componente conectado.
        - 'area': Área en píxeles.
        - 'center': Coordenadas (x, y) garantizadas dentro del área.
        - 'max_inscribed_radius': Distancia euclidiana máxima al borde (radio inscrito).
    """
    # Asegurar formato uint8 binario
    mask = (binary_mask > 0).astype(np.uint8) * 255

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask, connectivity=8
    )

    if num_labels <= 1:
        return np.zeros_like(mask), []

    # Extraer áreas omitiendo el fondo (índice 0)
    all_areas = stats[1:, cv2.CC_STAT_AREA].astype(np.float32)

    # Pre-filtrado de ruido microscópico
    valid_indices = np.where(all_areas >= min_absolute_pixels)[0]
    if len(valid_indices) == 0:
        return np.zeros_like(mask), []

    candidate_areas = all_areas[valid_indices]

    # 1. Determinación del umbral estadístico de área
    if method == "otsu" and len(candidate_areas) > 2:
        log_areas = np.log1p(candidate_areas)
        min_log, max_log = log_areas.min(), log_areas.max()

        if max_log > min_log:
            norm_log = np.uint8(255 * (log_areas - min_log) / (max_log - min_log))
            thresh_val, _ = cv2.threshold(
                norm_log, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )
            threshold_log = min_log + (thresh_val / 255.0) * (max_log - min_log)
            area_threshold = np.expm1(threshold_log)
        else:
            area_threshold = candidate_areas.min()

    elif method == "iqr":
        q1 = np.percentile(candidate_areas, 25)
        q3 = np.percentile(candidate_areas, 75)
        iqr = q3 - q1
        area_threshold = q3 + (iqr_k * iqr) if iqr > 0 else np.median(candidate_areas)

    else:
        # Fallback defensivo si hay muy pocas regiones para Otsu
        area_threshold = np.median(candidate_areas)

    # 2. Extracción de regiones y polos de inaccesibilidad
    filtered_mask = np.zeros_like(mask)
    regions_info = []

    for idx in valid_indices:
        label = idx + 1  # Compensar el índice del fondo
        area = int(stats[label, cv2.CC_STAT_AREA])

        if area >= area_threshold:
            comp_mask = np.uint8(labels == label)
            filtered_mask[comp_mask > 0] = 255

            # Transformada de distancia euclidiana
            dist_map = cv2.distanceTransform(comp_mask, cv2.DIST_L2, 5)
            _, max_val, _, max_loc = cv2.minMaxLoc(dist_map)

            regions_info.append(
                {
                    "label": label,
                    "area": area,
                    "center": max_loc,  # (columna_x, fila_y) dentro del área
                    "max_inscribed_radius": float(max_val),
                }
            )

    return filtered_mask, regions_info

In [ ]:
from typing import Dict, List, Tuple
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def plot_regions_with_centers(
    image: Image.Image,
    filtered_mask: np.ndarray,
    regions_info: List[Dict],
    show_mask: bool = True,
    overlay_color: Tuple[int, int, int] = (255, 30, 30),
    alpha: float = 0.4,
    point_color: str = "cyan",
    show_radius: bool = True,
) -> None:
    """Muestra los centros interiores sobre la imagen, con opción de activar o desactivar

    la máscara superpuesta.

    Parámetros:
    -----------
    show_mask : bool
        True superpone la silueta semitransparente; False muestra la imagen limpia con los puntos.
    show_radius : bool
        Dibuja el círculo inscrito máximo alrededor del punto interior.
    """
    img_np = np.array(image).copy()

    # Superponer máscara si show_mask está habilitado
    if show_mask and filtered_mask is not None:
        overlay = img_np.copy()
        overlay[filtered_mask == 255] = overlay_color
        canvas = cv2.addWeighted(img_np, 1 - alpha, overlay, alpha, 0)
    else:
        canvas = img_np

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(canvas)

    # Dibujar puntos y etiquetas
    for reg in regions_info:
        cx, cy = reg["center"]
        radius = reg["max_inscribed_radius"]

        # Punto interior (polo de inaccesibilidad)
        ax.scatter(
            cx, cy, c=point_color, edgecolors="black", s=60, zorder=5, marker="o"
        )

        # Círculo inscrito máximo
        if show_radius and radius > 2:
            circle = plt.Circle(
                (cx, cy),
                radius,
                color=point_color,
                fill=False,
                linestyle="--",
                linewidth=1.2,
                alpha=0.8,
            )
            ax.add_patch(circle)

        # Etiqueta con el número de componente
        ax.annotate(
            f"#{reg['label']}",
            (cx, cy),
            textcoords="offset points",
            xytext=(7, 7),
            color="white",
            fontsize=9,
            weight="bold",
            bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.6, lw=0),
        )

    title_suffix = "con máscara" if show_mask else "imagen limpia"
    ax.set_title(
        f"Regiones Salientes ({len(regions_info)} detectadas) — {title_suffix}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "facebook/dinov3-vitl16-pretrain-lvd1689m"

model, processor = load_backbone(model_id, device=device)


In [19]:
video_paths = [f"test_media/dolphin_{i:02d}.mp4" for i in range(4)]

In [20]:
frames = load_sampled_frames(video_path = video_paths[0],
                             n_frames = 5
                            )

NameError: name 'load_sampled_frames' is not defined

In [ ]:
final_mask, norm_score, cropped_img = run_dino_detector(raw_image=frames[0],
                                               model=model,
                                               processor=processor,
                                               resolution_scale=4.0,
                                               device = device
                                              )

In [ ]:
results=[]
for frame in frames:
    final_mask, norm_score, cropped_img = run_dino_detector(raw_image=frame,
                                                   model=model,
                                                   processor=processor,
                                                   resolution_scale=4.0,
                                                   device = device
                                                  )
    clean_mask, regions = extract_salient_regions(
        final_mask, method="otsu", min_absolute_pixels=30
    )
    results.append((cropped_img,clean_mask, regions))
    

In [ ]:
len(results)

In [ ]:
region = results[1]
plot_regions_with_centers(
    region[0],
    region[1],
    region[2],
    show_mask=True,
    show_radius=False
)

In [ ]:
suma = 0
for i in range(len(results)):
    suma += len(results[i][2])
promedio = suma / len(results)
print(promedio)
    